In [ ]:
import pandas as pd
import numpy as np
import csv
import os
import time
import matplotlib.pyplot as plt
from glob import glob

## Reading and Concatenating

### Original List

In [ ]:
df = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/reddit_comments_democrats.txt",
                 sep="|",
                 header=None
                 )
df

In [ ]:
df = df[0].tolist()
df_l = [i.split('/')[-2] for i in df]

In [ ]:
# dictionary of lists 
df_org = pd.DataFrame({'url': df, 'id': df_l})
df_org

In [ ]:
df_org[df_org['id']=='hkjrkb1'] #Climate

In [ ]:
df_org[df_org['id']=='hkjrkb1'] #BLM

In [ ]:
df_org[df_org['id']=='i0eirx4'] #BLM

In [ ]:
df_org[df_org['id']=='i0eirx4'] #republicans

In [ ]:
df_org[df_org['id']=='i0eirx4'] #Democrats

### Scrapped

In [ ]:
### DEFINIMOS UNA VARIABLE QUE NOS FACILITE LA RUTA DE LOS ARCHIVOS
program_path = r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/scrapped"
print(program_path)

In [ ]:
def ObtenerArchivos(ruta_actual,carpeta):
    """Esta función nos devuelve una lista con los archivos de una carpeta"""
    ruta_completa = os.path.join(ruta_actual,carpeta)
    archivos = glob(ruta_completa+"/*")
    return archivos

In [ ]:
files = ObtenerArchivos(program_path,'climate')
print(files)

In [ ]:
lista = [] ### LISTA AUXILIAR --> 

for archivo in files:
    if ".csv" in archivo:
        print("archivo: ",archivo)
        df = pd.read_csv(archivo,
                         sep="|")
        lista.append(df)
        
df_scrapped = pd.concat(lista,axis=0, sort = False)
df_scrapped

In [ ]:
df_scrapped.drop(columns=['score'], inplace= True)

In [ ]:
# Remove all duplicate rows
df_scrapped = df_scrapped.drop_duplicates(keep=False)
df_scrapped

### Datasets Length

In [ ]:
len(df_org)

In [ ]:
len(df_scrapped)

In [ ]:
len(df_org)-len(df_scrapped)

In [ ]:
df_org[df_org["id"].isna()]

In [ ]:
df_scrapped[df_scrapped["id"].isna()]

In [ ]:
df_scrapped[df_scrapped.duplicated(subset=['id'],keep=False)]

In [ ]:
df_scrapped[df_scrapped['id']=='hya4zm8']

### MERGE

In [ ]:
df_merged = df_org.merge(df_scrapped, left_on='id', right_on='id', how='left', indicator=True)
df_merged

In [ ]:
#df.to_excel(r"merged.xlsx",header=True, index=False)

In [ ]:
df_merged[df_merged.duplicated(subset=['id'],keep=False)]

In [ ]:
df_merged[df_merged.duplicated(subset=['id'],keep=False)]

In [ ]:
df_merged['_merge'].unique()

In [ ]:
len(df_merged[df_merged["author"].isna()])

In [ ]:
len(df_merged[df_merged["body"].isna()])#["author"].unique()

In [ ]:
len(df_merged[df_merged["body"].isin(["[deleted]","[removed]"])])#["author"].unique()

In [ ]:
len(df_merged) - (len(df_merged[df_merged["body"].isna()]) + len(df_merged[df_merged["body"].isin(["[deleted]","[removed]"])]))

In [ ]:
#df_merged[df_merged["author"].isna() & ~df_merged["body"].isin(["[deleted]","[removed]"]) & ~df_merged["body"].isna()]
df_merged = df_merged[df_merged["_merge"]=='both']
df_merged = df_merged[~df_merged["body"].isin(["[deleted]","[removed]"])]
df_merged = df_merged[~df_merged["body"].isna()]
df_merged

In [ ]:
len(df_merged[df_merged["author"].isna()])

In [ ]:
len(df_merged) - len(df_merged[df_merged["author"].isna()])

In [ ]:
df_merged = df_merged[~df_merged["author"].isna()]
df_merged.reset_index(drop=True, inplace=True)
df_merged

In [ ]:
#EXPORT
df_merged.to_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/list_scrap/climate/climate_all.csv",
          header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|")

In [ ]:
df = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/osfstorage-archive/list_scrap/climate/climate_all.csv",
                 sep="|")
df

### Stats

In [ ]:
author_count = pd.DataFrame(df_merged.groupby('body')['id'].nunique())

In [ ]:
author_count[author_count['id']>2]

In [ ]:
author_count = pd.DataFrame(df_merged.groupby('author')['id'].nunique())
author_count.reset_index(inplace=True)
author_count

In [ ]:
plt.hist(author_count['id'])

In [ ]:
author_count["id"].mean()

In [ ]:
author_count["id"].median()

In [ ]:
author_count.sort_values(by='id', ascending=False).tail(800)

In [ ]:
author_count[author_count["id"]>=author_count["id"].mean()]

In [ ]:
author_count[author_count["id"]<=300]

In [ ]:
author_count[author_count["id"]>=800]

In [ ]:
author_count[author_count["id"]>=900]